# Company AI Agent

Local Autonomous AI Agent for WhatsApp

- Arabic and English support
- Qwen3 via Ollama
- Excel / Google Sheets
- Customer memory
- RAG search
- Tool calling
- Controlled business operations
- WhatsApp Cloud API

In [ ]:
%pip install -q \
    requests \
    python-dotenv \
    flask \
    gspread \
    google-auth \
    openpyxl \
    rank-bm25 \
    sentence-transformers \
    apscheduler \
    nest-asyncio

In [ ]:
import os
import re
import json
import time
import hmac
import hashlib
import sqlite3
import logging
import threading
import requests

from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv

from flask import Flask, request

from openpyxl import load_workbook
from rank_bm25 import BM25Okapi

from google.oauth2.service_account import Credentials
import gspread

from concurrent.futures import ThreadPoolExecutor

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger("CompanyAgent")

print("Libraries loaded successfully")

In [ ]:
CONFIG = {
    # Local AI
    "OLLAMA_URL": "http://127.0.0.1:11434",
    "OLLAMA_MODEL": "qwen3:8b",

    # Data source: excel or google_sheets
    "SOURCE_TYPE": "excel",

    "EXCEL_FILE": "company.xlsx",
    "EXCEL_SHEET": None,

    "GOOGLE_SHEET_ID": "",
    "GOOGLE_CREDENTIALS": "service_account.json",

    # WhatsApp
    "WHATSAPP_TOKEN": "",
    "PHONE_NUMBER_ID": "",
    "WHATSAPP_VERIFY_TOKEN": "",
    "WHATSAPP_APP_SECRET": "",
    "WA_API_VERSION": "vXX.X",

    # Database
    "DATABASE": "company_agent.db",

    # Agent
    "MAX_TOOL_STEPS": 5,
    "MAX_HISTORY": 12,

    # Safety
    "MAX_DAILY_MESSAGES_PER_CUSTOMER": 100,
    "MAX_PARALLEL_TASKS": 4,

    # Search
    "SEARCH_RESULTS": 6
}

print("Configuration loaded")

In [ ]:
def check_ollama():

    url = CONFIG["OLLAMA_URL"] + "/api/tags"

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    models = response.json().get("models", [])

    print("Available models:")

    for model in models:
        print("-", model["name"])

    expected = CONFIG["OLLAMA_MODEL"]

    if not any(
        m["name"] == expected
        for m in models
    ):
        print(f"\nModel {expected} is not installed.")
        print(f"Run: ollama pull {expected}")
    else:
        print("\nModel is ready!")

check_ollama()

In [ ]:

def db_connect():
    conn = sqlite3.connect(
        CONFIG["DATABASE"],
        timeout=30
    )
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys=ON")
    return conn


def initialize_database():

    with db_connect() as db:

        db.execute("PRAGMA journal_mode=WAL")

        db.executescript("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            phone TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TEXT NOT NULL
        );

        CREATE INDEX IF NOT EXISTS idx_messages_phone
        ON messages(phone, id);

        CREATE TABLE IF NOT EXISTS customers (
            phone TEXT PRIMARY KEY,
            first_seen TEXT NOT NULL,
            last_seen TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS tickets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            phone TEXT NOT NULL,
            subject TEXT NOT NULL,
            details TEXT NOT NULL,
            status TEXT DEFAULT 'open',
            created_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS orders (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            phone TEXT NOT NULL,
            details TEXT NOT NULL,
            status TEXT DEFAULT 'pending_review',
            created_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS processed_messages (
            message_id TEXT PRIMARY KEY,
            status TEXT NOT NULL,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS tool_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            phone TEXT NOT NULL,
            tool_name TEXT NOT NULL,
            result TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        """)

    print("Database initialized")


def current_time():
    return datetime.now(timezone.utc).isoformat()


def save_message(phone, role, content):

    now = current_time()

    with db_connect() as db:

        db.execute("""
            INSERT INTO messages
            (phone, role, content, created_at)
            VALUES (?, ?, ?, ?)
        """, (phone, role, content, now))

        db.execute("""
            INSERT INTO customers
            (phone, first_seen, last_seen)
            VALUES (?, ?, ?)
            ON CONFLICT(phone)
            DO UPDATE SET last_seen=excluded.last_seen
        """, (phone, now, now))


def get_history(phone, limit=None):

    limit = limit or CONFIG["MAX_HISTORY"]

    with db_connect() as db:

        rows = db.execute("""
            SELECT role, content
            FROM messages
            WHERE phone=?
            ORDER BY id DESC
            LIMIT ?
        """, (phone, limit)).fetchall()

    return [
        {
            "role": row["role"],
            "content": row["content"]
        }
        for row in reversed(rows)
    ]


def create_ticket(phone, subject, details):

    with db_connect() as db:

        cur = db.execute("""
            INSERT INTO tickets
            (phone, subject, details, created_at)
            VALUES (?, ?, ?, ?)
        """, (phone, subject, details, current_time()))

        return cur.lastrowid


def create_order_draft(phone, details):

    with db_connect() as db:

        cur = db.execute("""
            INSERT INTO orders
            (phone, details, created_at)
            VALUES (?, ?, ?)
        """, (phone, details, current_time()))

        return cur.lastrowid


def log_tool(phone, name, result):

    with db_connect() as db:

        db.execute("""
            INSERT INTO tool_logs
            (phone, tool_name, result, created_at)
            VALUES (?, ?, ?, ?)
        """, (
            phone,
            name,
            json.dumps(result, ensure_ascii=False),
            current_time()
        ))


initialize_database()
